# Local smoke test — Ray Data (Topic 1)

## Purpose

This notebook runs **`scale_data.py` in your workbench pod** — the same script Topic 2 submits via **`cluster.job_client`** on a KubeRay cluster.

We do this to:

1. **Validate the environment** — repo cloned, paths correct, Ray/pandas/pyarrow available — before OpenShift API auth, LocalQueues, and RayClusters (Topic 2).
2. **Separate failures** — if this notebook succeeds but Topic 2 fails, suspect **platform configuration** (token, queue, quota, RayCluster), not the lab Python.
3. **Introduce Ray Data on a tiny dataset** — read CSV, `map_batches`, write Parquet — without CodeFlare or cluster lifecycle in the same step.

## What actually runs (not distributed)

The notebook does **not** create a Kubernetes RayCluster. It simply runs the Python script with `subprocess`:

```python
python extras/scripts/scale_data.py
```

Inside that script, **Ray Data** calls (`ray.data.read_csv`, `map_batches`, etc.) auto-start a **local, single-node** Ray runtime in the workbench pod. There is no KubeRay cluster, no worker pods, and no multi-node scheduling — on this tiny Iris dataset it is effectively one process in one pod.

| | Topic 1 (here) | Topic 2+ |
|---|---|---|
| How | Run script in workbench | `Cluster` + `job_client` via CodeFlare |
| Ray | Local single-node in pod | KubeRay head + workers |
| Distributed? | No | Yes |

## Required?

- **Recommended** for first-time runs and when validating a new cluster.
- **Optional** if you are comfortable skipping straight to `02-ray-data-job-client.ipynb`.

If local Ray fails due to workbench memory limits, you may still proceed to Topic 2 — the main workshop path is the KubeRay cluster, not local Ray in the workbench.

## What this does not prove

- Kueue admission or LocalQueue configuration
- CodeFlare authentication or `job_client` submission
- Multi-node or production-scale Ray

Those are Topics 2–4. No OpenShift API token is needed for this notebook.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

# JupyterLab cwd is this notebook's folder (.../extras/notebooks).
sys.path.insert(0, str(Path.cwd()))
from workshop_bootstrap import setup_workshop_path

REPO_ROOT = setup_workshop_path()
SCRIPT = REPO_ROOT / "extras/scripts/scale_data.py"
DATA = REPO_ROOT / "extras/data/iris.csv"
OUT = REPO_ROOT / "tmp/iris-local"

print(f"Repo root: {REPO_ROOT}")

env = os.environ.copy()
env["INPUT_PATH"] = str(DATA)
env["OUTPUT_DIR"] = str(OUT)
OUT.mkdir(parents=True, exist_ok=True)

subprocess.run([sys.executable, str(SCRIPT)], check=True, env=env)
print("Parquet files:", list(OUT.glob("*.parquet")))